# Deep Q-Network (DQN)

Il codice del capitolo [«Deep Q-Network (DQN)»](https://book.paithon.it/main/DeepReinforcementLearning/dqn.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.%pip install -q numpy torch torchvision

## Deep Q-Network (DQN)

[Leggi la pagina](https://book.paithon.it/main/DeepReinforcementLearning/dqn.html)


### Rete-target


In [ ]:
from torch import nndef crea_q_network(n_azioni):    # ingresso (4, 84, 84): 4 fotogrammi impilati (per cogliere il movimento)    return nn.Sequential(        nn.Conv2d(4, 32, kernel_size=8, stride=4),        nn.ReLU(),        nn.Conv2d(32, 64, kernel_size=4, stride=2),        nn.ReLU(),        nn.Conv2d(64, 64, kernel_size=3, stride=1),        nn.ReLU(),        nn.Flatten(),        nn.Linear(64 * 7 * 7, 512),        nn.ReLU(),        nn.Linear(512, n_azioni),  # un valore Q per azione, nessuna attivazione    )

*Frammento illustrativo: nel libro mostra la forma, qui non si esegue.*

```python

import torch

# minibatch pescato a caso dalla memoria di replay
s, a, r, s_next, fine = replay.campiona(batch=32)

with torch.no_grad():                              # il bersaglio non si deriva
    q_next = target_net(s_next).max(dim=1).values  # max_a' Q(s', a'; theta^-)
bersaglio = r + gamma * q_next * (1 - fine)        # se terminale, resta solo r
```


## Metodi a gradiente di policy

[Leggi la pagina](https://book.paithon.it/main/DeepReinforcementLearning/policy-gradient.html)


### In pratica: le visite si concentrano


In [ ]:
import mathimport numpy as np# Un albero giocattolo: profondità 4, due mosse per nodo, 16 foglie.# I valori delle foglie li conosciamo, così sappiamo qual è la risposta giusta.PROFONDITA, RAMI = 4, 2rng = np.random.default_rng(7)valori_foglie = rng.uniform(0, 1, RAMI ** PROFONDITA)valori_foglie[6] = 0.98                      # la foglia buona, nascosta in mezzomigliore = int(valori_foglie.argmax())PRIME = (RAMI ** PROFONDITA - 1) // (RAMI - 1)   # indice della prima fogliadef figli(nodo):    return [nodo * RAMI + 1 + k for k in range(RAMI)]def foglia(nodo):    return nodo >= PRIMEN, W = {0: 0}, {0: 0}                        # visite e somma dei ritornidef uct(nodo, c=1.4):    """UCB1 applicato a un bivio dell'albero: è la formula della sezione bandit."""    padre = N[nodo]    def punteggio(f):        if N.get(f, 0) == 0:            return float("inf")              # mai provato: massimamente urgente        return W[f] / N[f] + c * math.sqrt(math.log(padre) / N[f])    return max(figli(nodo), key=punteggio)def simula(nodo):    """Discesa a caso fino a una foglia: la stima grezza di questo nodo."""    while not foglia(nodo):        nodo = int(rng.choice(figli(nodo)))    return valori_foglie[nodo - PRIME]for _ in range(2000):    nodo, cammino = 0, [0]    while not foglia(nodo) and all(N.get(f, 0) > 0 for f in figli(nodo)):        nodo = uct(nodo)                                          # 1. SELEZIONE        cammino.append(nodo)    if not foglia(nodo):        nodo = next(f for f in figli(nodo) if N.get(f, 0) == 0)   # 2. ESPANSIONE        cammino.append(nodo)        N[nodo] = W[nodo] = 0    ritorno = simula(nodo)                                        # 3. SIMULAZIONE    for n in cammino:                                             # 4. RISALITA        N[n] += 1        W[n] += ritornoprint("visite ai due rami dalla radice:", [N[f] for f in figli(0)])print("valore medio dei due rami      :", [round(float(W[f] / N[f]), 3)                                           for f in figli(0)])visite_foglie = np.array([N.get(PRIME + i, 0) for i in range(RAMI ** PROFONDITA)])print(f"foglia migliore: {migliore} (valore {valori_foglie[migliore]:.2f})")print(f"quota delle visite andata lì: "      f"{visite_foglie[migliore] / visite_foglie.sum():.1%}")print(f"tirando a caso sarebbe stata: {1 / len(valori_foglie):.1%}")

## Controllo continuo: DDPG, TD3, SAC

[Leggi la pagina](https://book.paithon.it/main/DeepReinforcementLearning/controllo-continuo.html)


### Lo scheletro dell'aggiornamento, in PyTorch


*Frammento illustrativo: nel libro mostra la forma, qui non si esegue.*

```python

import torch
import torch.nn.functional as F

# reti gia definite: attore mu(s), critico q_net(s, a) e le loro copie target
# ottimizzatori: opt_critico (parametri di q_net), opt_attore (parametri di mu)
# minibatch dal replay buffer, come in DQN: tensori s, a, r, s_next, fine

# --- bersaglio di Bellman: non si deriva, usa le reti target ---
with torch.no_grad():
    a_next = mu_target(s_next)                     # azione greedy dell'attore target
    q_next = q_target(s_next, a_next)              # Q^-(s', mu^-(s'))
    y = r + gamma * q_next * (1 - fine)            # se terminale resta solo r

# --- aggiornamento del critico: avvicina Q(s, a) al bersaglio ---
q = q_net(s, a)                                    # Q sulle azioni realmente eseguite
perdita_critico = F.mse_loss(q, y)
opt_critico.zero_grad()
perdita_critico.backward()
opt_critico.step()

# --- aggiornamento dell'attore: sali lungo il gradiente del critico ---
perdita_attore = -q_net(s, mu(s)).mean()           # massimizza Q(s, mu(s))
opt_attore.zero_grad()
perdita_attore.backward()                          # il gradiente scorre da Q dentro mu
opt_attore.step()

# --- aggiornamento morbido (Polyak) delle reti target ---
with torch.no_grad():
    for p, p_t in zip(q_net.parameters(), q_target.parameters()):
        p_t.mul_(1 - tau).add_(tau * p)
    for p, p_t in zip(mu.parameters(), mu_target.parameters()):
        p_t.mul_(1 - tau).add_(tau * p)
```


## Reinforcement learning basato su modello

[Leggi la pagina](https://book.paithon.it/main/DeepReinforcementLearning/model-based.html)


### Dyna: intrecciare il vero e l'immaginato


In [ ]:
import numpy as nprng = np.random.default_rng(0)# Ambiente: corridoio di 6 stati; l'obiettivo e' lo stato 5 (assorbente).# Azioni: 0 = sinistra, 1 = destra. Ricompensa +1 solo entrando nell'obiettivo.n_stati, n_azioni, goal = 6, 2, 5def passo(s, a):    s2 = min(s + 1, goal) if a == 1 else max(s - 1, 0)    r = 1.0 if s2 == goal else 0.0    return s2, r, (s2 == goal)def scelta_greedy(q):              # argmax con i pareggi rotti a caso    return int(rng.choice(np.flatnonzero(q == q.max())))Q = np.zeros((n_stati, n_azioni))modello = {}                       # (s, a) -> (r, s2): la dinamica APPRESAalpha, gamma, eps, n_plan = 0.1, 0.95, 0.1, 20for _ in range(30):                # appena 30 episodi reali    s = 0    for _ in range(100):        a = int(rng.integers(n_azioni)) if rng.random() < eps else scelta_greedy(Q[s])        s2, r, fine = passo(s, a)        # 1) aggiornamento dall'esperienza REALE        Q[s, a] += alpha * (r + gamma * Q[s2].max() - Q[s, a])        modello[(s, a)] = (r, s2)  # memorizza la transizione osservata        # 2) n passi di PLANNING su transizioni gia' viste (esperienza immaginata)        viste = list(modello.keys())        for _ in range(n_plan):            sp, ap = viste[rng.integers(len(viste))]            rp, s2p = modello[(sp, ap)]            Q[sp, ap] += alpha * (rp + gamma * Q[s2p].max() - Q[sp, ap])        s = s2        if fine:            breakpolicy = np.argmax(Q, axis=1)      # ci aspettiamo 1 (destra) ovunqueprint("Policy appresa (0=sx, 1=dx):", policy[:goal].tolist())

### Dreamer: allenare la policy nel sogno


In [ ]:
import torchfrom torch import nnclass ModelloDinamica(nn.Module):    """Modello appreso: da (stato, azione) predice stato successivo e ricompensa."""    def __init__(self, dim_s, dim_a, dim_h=200):        super().__init__()        self.corpo = nn.Sequential(            nn.Linear(dim_s + dim_a, dim_h), nn.SiLU(),            nn.Linear(dim_h, dim_h), nn.SiLU(),        )        self.testa_stato = nn.Linear(dim_h, dim_s)    # variazione dello stato        self.testa_ricompensa = nn.Linear(dim_h, 1)   # ricompensa predetta    def forward(self, s, a):                          # s: (B, dim_s), a: (B, dim_a)        h = self.corpo(torch.cat([s, a], dim=-1))        s_succ = s + self.testa_stato(h)              # residuo: predice il cambiamento        r = self.testa_ricompensa(h).squeeze(-1)      # (B,)        return s_succ, r# Rollout BREVE immaginato (stile MBPO): parte da stati reali, pochi passi.modello = ModelloDinamica(dim_s=4, dim_a=1)policy = nn.Sequential(nn.Linear(4, 1), nn.Tanh())    # policy giocattolos = torch.randn(32, 4)                                # 32 stati REALI dal bufferfor _ in range(3):                                    # orizzonte corto: 3 passi    a = policy(s)                                     # (32, 1)    s, r = modello(s, a)                              # transizioni SINTETICHE

## Imparare guardando: imitazione e clonazione comportamentale

[Leggi la pagina](https://book.paithon.it/main/DeepReinforcementLearning/imitazione.html)


### In pratica: l'errore per passo è zero, e il sistema finisce nel fosso


In [ ]:
import torchimport torch.nn as nnimport torch.nn.functional as Ftorch.manual_seed(0)INSTABILE, RUMORE, ORIZZONTE, RAFFICA = 1.25, 0.02, 40, 4.0def esperto(s):    """Un controllore vero: sa cosa fare ovunque, anche lontano da zero."""    return -(INSTABILE - 0.85) * sdef episodio(politica, s0, g, raffica=False):    s, stati, azioni = s0.clone(), [], []    for t in range(ORIZZONTE):        if raffica and t == 15:            s = s + RAFFICA                 # una folata che l'esperto non ha mai preso        with torch.no_grad():            a = politica(s)        stati.append(s.clone()); azioni.append(esperto(s))   # l'esperto etichetta        s = INSTABILE * s + a + RUMORE * torch.randn(s.shape, generator=g)    return torch.cat(stati), torch.cat(azioni), sg = torch.Generator().manual_seed(1)avvio = lambda n: torch.randn(n, 1, generator=g) * 0.3# le dimostrazioni: l'esperto è bravo, quindi non si allontana mai da zeroS_dim, A_dim, _ = episodio(esperto, avvio(64), g)print(f"stati dimostrati: fra {S_dim.min():+.2f} e {S_dim.max():+.2f}")politica = nn.Sequential(nn.Linear(1, 64), nn.Tanh(), nn.Linear(64, 64),                         nn.Tanh(), nn.Linear(64, 1))def allena(S, A, passi=2000):    ott = torch.optim.Adam(politica.parameters(), lr=3e-3)    for _ in range(passi):        ott.zero_grad(); F.mse_loss(politica(S), A).backward(); ott.step()allena(S_dim, A_dim)with torch.no_grad():    print(f"errore per singolo passo sugli stati dimostrati: "          f"{F.mse_loss(politica(S_dim), A_dim).item():.6f}  (praticamente perfetto)")    fuori = torch.tensor([[3.0]])    print(f"ma a s=3,0 l'esperto direbbe {esperto(fuori).item():+.3f} "          f"e la clonazione dice {politica(fuori).item():+.3f}")_, _, fe = episodio(esperto, avvio(64), g, raffica=True)_, _, fc = episodio(politica, avvio(64), g, raffica=True)print(f"\ndopo la folata: |stato| finale esperto {fe.abs().mean():.3f}, "      f"clonazione {fc.abs().mean():.1f}")# DAgger: si aggiungono gli stati in cui è finita LA POLITICA, etichettati# dall'esperto. Sono proprio quelli che le dimostrazioni non contenevano.S_tot, A_tot = S_dim, A_dimfor giro in (1, 2, 3):    S_v, A_v, _ = episodio(politica, avvio(64), g, raffica=True)    S_tot, A_tot = torch.cat([S_tot, S_v]), torch.cat([A_tot, A_v])    allena(S_tot, A_tot, passi=1500)    _, _, f = episodio(politica, avvio(64), g, raffica=True)    print(f"dopo il giro {giro} di DAgger: |stato| finale {f.abs().mean():.3f}")

## Offline reinforcement learning: imparare da dati fissi

[Leggi la pagina](https://book.paithon.it/main/DeepReinforcementLearning/offline-rl.html)


### Il buco nero delle azioni mai viste


In [ ]:
import numpy as nprng = np.random.default_rng(0)# Valore-azione "vero": una parabola con l'ottimo in a = 0.3def Q_vero(a):    return -(a - 0.3) ** 2# La policy comportamentale ha esplorato solo azioni "prudenti" in [-1.0, -0.2]:# il dataset NON contiene mai l'azione ottima.a_dati = rng.uniform(-1.0, -0.2, size=40)q_dati = Q_vero(a_dati) + rng.normal(0, 0.01, size=a_dati.shape)# Stimiamo Q con un modello flessibile (polinomio di grado 5) sui soli dati.coeff = np.polyfit(a_dati, q_dati, deg=5)Q_stima = np.poly1d(coeff)azioni = np.linspace(-1, 1, 201)                 # candidate su TUTTO lo spazio# (1) max naive: nessun vincolo, come nel Q-learning off-policy classicoa_naive = azioni[np.argmax(Q_stima(azioni))]print(f"naive     : a*={a_naive:+.2f}  Q_stimato={Q_stima(a_naive):+.2f}  "      f"Q_vero={Q_vero(a_naive):+.2f}")# (2) max vincolato al supporto dei dati (idea alla base di BCQ)in_supp = (azioni >= a_dati.min()) & (azioni <= a_dati.max())a_vinc = azioni[in_supp][np.argmax(Q_stima(azioni[in_supp]))]print(f"vincolato : a*={a_vinc:+.2f}  Q_stimato={Q_stima(a_vinc):+.2f}  "      f"Q_vero={Q_vero(a_vinc):+.2f}")

### IQL: non guardare mai fuori dai dati


In [ ]:
import torchdef expectile_loss(q, v, tau=0.8):    # regressione expectile: residui positivi pesati di piu (tau > 0.5)    diff = q - v    peso = torch.where(diff > 0, tau, 1.0 - tau)    return (peso * diff.pow(2)).mean()

## Esplorazione e ricompensa: curiosità, sparsità, reward hacking

[Leggi la pagina](https://book.paithon.it/main/DeepReinforcementLearning/esplorazione-e-ricompensa.html)


### Bonus di novità: premiare ciò che si visita di rado


In [ ]:
import numpy as np# Conteggi di visita di 6 stati in un piccolo ambiente tabellarevisite = np.array([120, 40, 5, 0, 200, 1])# Bonus di novità count-based: più raro lo stato, più alto il bonus.beta = 0.5bonus = beta / np.sqrt(visite + 1)   # +1 evita la divisione per zerofor s, (n, b) in enumerate(zip(visite, bonus)):    print(f"stato {s}: visite={n:3d}  bonus={b:.3f}")

### Curiosità intrinseca: la sorpresa come ricompensa


In [ ]:
import torchimport torch.nn as nn# RND: due reti con la stessa architettura.# target: pesi casuali FISSI, mai addestrati; predictor: impara a imitarla.def crea_rete(dim_stato, dim_feature=64):    return nn.Sequential(        nn.Linear(dim_stato, 128), nn.ReLU(),        nn.Linear(128, dim_feature),    )target = crea_rete(dim_stato=16)predictor = crea_rete(dim_stato=16)# La rete target è congelata: non le passa mai gradiente.for p in target.parameters():    p.requires_grad_(False)optimizer = torch.optim.Adam(predictor.parameters(), lr=1e-3)def ricompensa_intrinseca(stati):    with torch.no_grad():        obiettivo = target(stati)            # output della rete casuale fissa    previsione = predictor(stati)    # errore per-stato = novità: alto sugli stati poco visti    return (previsione - obiettivo).pow(2).mean(dim=1)stati = torch.randn(8, 16)r_int = ricompensa_intrinseca(stati)         # bonus di novità del batch# Il predictor si allena a ridurre l'errore sugli stati che l'agente visita:# così quegli stati, in futuro, saranno meno "sorprendenti".loss = r_int.mean()optimizer.zero_grad()loss.backward()optimizer.step()